In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"D:\CreditIQ\data\application_train.csv")
print(f"Raw: {df.shape}")

Raw: (307511, 122)


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(r"D:\CreditIQ\data\application_train.csv")
print(f"Raw: {df.shape}")

# 1. Drop columns with >40% missing and weak correlation
drop_cols = [col for col in df.columns 
             if df[col].isnull().mean() > 0.40 and col not in ['EXT_SOURCE_1']]
df = df.drop(columns=drop_cols)

# 2. Fix DAYS_EMPLOYED sentinel
df['EMPLOYED_ANOMALY'] = (df['DAYS_EMPLOYED'] == 365243).astype(int)
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)

# 3. Fill + Flag important columns with significant missing
for col in ['EXT_SOURCE_1', 'EXT_SOURCE_3', 'DAYS_EMPLOYED']:
    df[f'{col}_MISSING'] = df[col].isnull().astype(int)
    df[col] = df[col].fillna(df[col].median())

# 4. Fill remaining missing
num_cols = df.select_dtypes(include='number').columns
cat_cols = df.select_dtypes(include='object').columns
df[num_cols] = df[num_cols].fillna(df[num_cols].median())
df[cat_cols] = df[cat_cols].fillna(df[cat_cols].mode().iloc[0])

print(f"Clean: {df.shape}")
print(f"Missing: {df.isnull().sum().sum()}")

Raw: (307511, 122)


C:\Users\athar\AppData\Local\Temp\ipykernel_8188\1700410674.py:23: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns


Clean: (307511, 78)
Missing: 0


In [3]:
df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL']
df['AGE_YEARS'] = df['DAYS_BIRTH'] / -365
df['DTI_RATIO'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL']

In [4]:
print("CREDIT_INCOME_RATIO:")
print(df.groupby('TARGET')['CREDIT_INCOME_RATIO'].median())
print()
print("DTI_RATIO:")
print(df.groupby('TARGET')['DTI_RATIO'].median())

CREDIT_INCOME_RATIO:
TARGET
0    3.266653
1    3.253143
Name: CREDIT_INCOME_RATIO, dtype: float64

DTI_RATIO:
TARGET
0    0.162280
1    0.169294
Name: DTI_RATIO, dtype: float64


In [9]:
df['EMPLOYED_YEARS'] = df['DAYS_EMPLOYED'] / -365
df['INCOME_PER_FAMILY'] = df['AMT_INCOME_TOTAL'] / df['CNT_FAM_MEMBERS']
df['CREDIT_GOODS_RATIO'] = df['AMT_CREDIT'] / df['AMT_GOODS_PRICE']
df['EXT_SOURCE_MEAN'] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].mean(axis=1)

# Check all engineered features against TARGET
new_features = ['CREDIT_INCOME_RATIO', 'DTI_RATIO', 'AGE_YEARS', 'EMPLOYED_YEARS',
                'INCOME_PER_FAMILY', 'CREDIT_GOODS_RATIO', 'EXT_SOURCE_MEAN']

for feat in new_features:
    corr = df[feat].corr(df['TARGET'])
    print(f"{feat}: {corr:.4f}")

CREDIT_INCOME_RATIO: -0.0077
DTI_RATIO: 0.0143
AGE_YEARS: -0.0782
EMPLOYED_YEARS: -0.0634
INCOME_PER_FAMILY: -0.0066
CREDIT_GOODS_RATIO: 0.0685
EXT_SOURCE_MEAN: -0.2208


In [10]:
bureau = pd.read_csv(r"D:\CreditIQ\data\bureau.csv")
print(bureau.shape)
print(bureau.head())

(1716428, 17)
   SK_ID_CURR  SK_ID_BUREAU CREDIT_ACTIVE CREDIT_CURRENCY  DAYS_CREDIT  \
0      215354       5714462        Closed      currency 1         -497   
1      215354       5714463        Active      currency 1         -208   
2      215354       5714464        Active      currency 1         -203   
3      215354       5714465        Active      currency 1         -203   
4      215354       5714466        Active      currency 1         -629   

   CREDIT_DAY_OVERDUE  DAYS_CREDIT_ENDDATE  DAYS_ENDDATE_FACT  \
0                   0               -153.0             -153.0   
1                   0               1075.0                NaN   
2                   0                528.0                NaN   
3                   0                  NaN                NaN   
4                   0               1197.0                NaN   

   AMT_CREDIT_MAX_OVERDUE  CNT_CREDIT_PROLONG  AMT_CREDIT_SUM  \
0                     NaN                   0         91323.0   
1                   

In [11]:
# Bureau features
bureau = pd.read_csv(r"D:\CreditIQ\data\bureau.csv")

bureau_agg = bureau.groupby('SK_ID_CURR').agg({
    'SK_ID_BUREAU': 'count',
    'AMT_CREDIT_SUM': 'sum',
    'CREDIT_DAY_OVERDUE': 'max',
    'AMT_CREDIT_SUM_DEBT': 'sum'
}).rename(columns={
    'SK_ID_BUREAU': 'PREV_LOAN_COUNT',
    'AMT_CREDIT_SUM': 'PREV_TOTAL_CREDIT',
    'CREDIT_DAY_OVERDUE': 'PREV_MAX_OVERDUE',
    'AMT_CREDIT_SUM_DEBT': 'PREV_TOTAL_DEBT'
})

# Count active loans separately
active_loans = bureau[bureau['CREDIT_ACTIVE'] == 'Active'].groupby('SK_ID_CURR').size()
active_loans.name = 'PREV_ACTIVE_LOANS'

# Merge into main df
df = df.merge(bureau_agg, on='SK_ID_CURR', how='left')
df = df.merge(active_loans, on='SK_ID_CURR', how='left')

# Fill NaN for applicants with no bureau history
bureau_cols = ['PREV_LOAN_COUNT', 'PREV_TOTAL_CREDIT', 'PREV_MAX_OVERDUE', 
               'PREV_TOTAL_DEBT', 'PREV_ACTIVE_LOANS']
df[bureau_cols] = df[bureau_cols].fillna(0)

print(f"Shape after bureau merge: {df.shape}")

Shape after bureau merge: (307511, 90)


In [12]:
for col in bureau_cols:
    corr = df[col].corr(df['TARGET'])
    print(f"{col}: {corr:.4f}")

PREV_LOAN_COUNT: -0.0100
PREV_TOTAL_CREDIT: -0.0180
PREV_MAX_OVERDUE: 0.0044
PREV_TOTAL_DEBT: 0.0019
PREV_ACTIVE_LOANS: 0.0436


In [13]:
# Save cleaned dataset
df.to_csv(r"D:\CreditIQ\data\application_clean.csv", index=False)
print(f"Saved: {df.shape}")
print(f"Features: {df.shape[1] - 2} (excluding SK_ID_CURR and TARGET)")

Saved: (307511, 90)
Features: 88 (excluding SK_ID_CURR and TARGET)
